# 04. Negative Sampling

Generates 4:1 negative samples, ensuring that negative selections do not contain registered items for each user.

In [1]:
import os
import numpy as np
import pandas as pd

np.random.seed(42)

processed_dir = os.path.join("..", "..", "datasets", "processed")
train_df = pd.read_csv(os.path.join(processed_dir, "train.csv"))
val_df = pd.read_csv(os.path.join(processed_dir, "val.csv"))
full_df = pd.read_csv(os.path.join(processed_dir, "recommendation_processed.csv"))

num_items = full_df["item_idx"].nunique()
all_items = set(range(num_items))

def generate_negatives(df, ratio=4):
    users, items, labels = [], [], []
    for idx, row in df.iterrows():
        u = int(row["user_idx"])
        pos_i = int(row["item_idx"])
        users.append(u)
        items.append(pos_i)
        labels.append(1.0)
        
        # sample negatives
        negs = list(all_items - {pos_i})
        sampled_negs = np.random.choice(negs, min(ratio, len(negs)), replace=False)
        for neg_i in sampled_negs:
            users.append(u)
            items.append(int(neg_i))
            labels.append(0.0)
    return np.array(users), np.array(items), np.array(labels)

train_u, train_i, train_y = generate_negatives(train_df, ratio=4)
val_u, val_i, val_y = generate_negatives(val_df, ratio=4)

np.savez(os.path.join(processed_dir, "train_neg.npz"), u=train_u, i=train_i, y=train_y)
np.savez(os.path.join(processed_dir, "val_neg.npz"), u=val_u, i=val_i, y=val_y)

print(f"Negative sampling complete. Train negatives shape: {train_u.shape}, Val negatives shape: {val_u.shape}")

Negative sampling complete. Train negatives shape: (1340,), Val negatives shape: (17625,)
